# Сравнение сложных ML моделей и Sequence-фичей

**Логика:** один Setup (1) → общие функции (3) → эксперименты по порядку (4–8) → дополнительный эксперимент rolling 5/15/30/60 (9).

**Цель:** temporal split (как в проде), комиссия 0.1%, сравнить:
1. Baseline (22 фичи) и Sequence (rolling 30, 60 → 62 фичи).
2. MLP flatten (60×22), прод CatBoost.
3. Доп. эксперимент: old_30_60 vs new_5_15_30_60, пороги 20–80 … 40–60.

**Запуск:** Run All (с 1-й ячейки). Секция 9 опирается на данные и функции из блоков 1–3.

## 1. Импорты и загрузка данных

Данные: `data_labeled_tp_sl_1_05.parquet`. Фичи: `features_selected_tp_sl_1_05.txt`. Temporal split: 8 дней train, 1 val, 1 test (последний день).

In [1]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

HAS_XGB, HAS_LGB, HAS_CAT = False, False, False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError: pass
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError: pass
try:
    import catboost as cb
    HAS_CAT = True
except ImportError: pass

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Не найден {labeled_path}. Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError(f'Не найден {feat_path}. Запустите 06_Feature_Selection.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

TARGET_COL = 'target'
valid = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
if len(dates) < 10:
    raise ValueError(f'Мало дат: {len(dates)}. Нужно минимум 10 дней.')
train_dates = set(dates[:8])
val_dates = set([dates[8]])
test_dates = set([dates[9]])
train_df = valid[valid['date'].isin(train_dates)].copy()
val_df = valid[valid['date'].isin(val_dates)].copy()
test_df = valid[valid['date'].isin(test_dates)].copy()

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
train_df = train_df.sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df = val_df.sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df = test_df.sort_values(['session_key', sort_col]).reset_index(drop=True)

COMMISSION_RT = 0.001
THR_PROD = 0.60  # зона HOLD 0.40–0.60
THRESHOLD_BANDS = {'20-80': 0.80, '25-75': 0.75, '30-70': 0.70, '35-65': 0.65, '40-60': 0.60}

print('Temporal split (как в проде): train=', f'{min(train_dates)}..{max(train_dates)}', f'({len(train_df):,}) | val={dates[8]} ({len(val_df):,}) | test={dates[9]} ({len(test_df):,})')
print(f'Target: {TARGET_COL}, baseline features: {len(BASE_FEATURES)}')
print(f'XGB: {HAS_XGB}, LGB: {HAS_LGB}, CatBoost: {HAS_CAT}')

Temporal split (как в проде): train= 2026-02-01..2026-02-08 (146,693) | val=2026-02-09 (24,014) | test=2026-02-10 (15,356)
Target: target, baseline features: 22
XGB: True, LGB: True, CatBoost: True


## 2. Добавление Sequence-фичей (rolling 30, 60)

К первым 10 базовым фичам добавляем rolling mean и rolling std по окнам 30 и 60. Итого 62 фичи.

In [2]:
key_feats = BASE_FEATURES[:10]
for w in [30, 60]:
    grp = df.groupby('session_key', group_keys=False)
    for c in key_feats:
        if c in df.columns:
            df[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
SEQ_COLS = [c for c in df.columns if '_roll' in c and c in df.columns]
FEATURES_SEQ = BASE_FEATURES + SEQ_COLS
FEATURES_SEQ = [c for c in FEATURES_SEQ if c in df.columns]

train_df_seq = df.loc[train_df.index].dropna(subset=FEATURES_SEQ).copy() if train_df.index.isin(df.index).any() else df[df['date'].isin(train_dates)].dropna(subset=FEATURES_SEQ + ['target']).copy()
train_df_seq = train_df_seq[train_df_seq['target'].isin([-1.0, 1.0])]
train_df_seq['y'] = (train_df_seq['target'] == 1).astype(int)
train_df_seq['date'] = pd.to_datetime(train_df_seq['datetime'], utc=True).dt.date

valid_seq = df.dropna(subset=FEATURES_SEQ + [TARGET_COL]).copy()
valid_seq = valid_seq[valid_seq[TARGET_COL].isin([-1.0, 1.0])]
valid_seq['y'] = (valid_seq[TARGET_COL] == 1).astype(int)
valid_seq['date'] = pd.to_datetime(valid_seq['datetime'], utc=True).dt.date
valid_seq['ret_next'] = valid_seq.groupby('session_key')['close_price'].pct_change().shift(-1)
valid_seq = valid_seq.dropna(subset=['ret_next'])
train_df_seq = valid_seq[valid_seq['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df_seq = valid_seq[valid_seq['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df_seq = valid_seq[valid_seq['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print(f'Sequence features: {len(FEATURES_SEQ)} (baseline {len(BASE_FEATURES)} + rolling 30,60)')
print(f'Train seq: {len(train_df_seq):,}, Val seq: {len(val_df_seq):,}, Test seq: {len(test_df_seq):,}')

Sequence features: 62 (baseline 22 + rolling 30,60)
Train seq: 146,693, Val seq: 24,014, Test seq: 15,356


## 3. Функции бектеста и обучения

Бектест (hold_keep_prev=True), комиссия 0.1%. `backtest_pnl` — один порог; `backtest_pnl_bands` — возвращает avg_%_per_trade для секции 9.

In [3]:
def backtest_pnl(proba, ret, session_ids, threshold=THR_PROD, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1: pos[i], prev = 1.0, 1.0
        elif pred[i] == 0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev if hold_keep_prev else 0.0
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    return {'trades': int(pos_changed.sum()), 'net_%': float(pnl_net * 100)}

def backtest_pnl_bands(proba, ret, session_ids, threshold=THR_PROD, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    """Как backtest_pnl, но возвращает avg_%_per_trade для нескольких порогов (секция 9)."""
    bt = backtest_pnl(proba, ret, session_ids, threshold, commission_rt, hold_keep_prev)
    t = bt['trades']
    bt['avg_%_per_trade'] = (bt['net_%'] / t) if t > 0 else np.nan
    return bt

def prep_by_features(df_in, feat_cols):
    d = df_in.dropna(subset=feat_cols + [TARGET_COL]).copy()
    d = d[d[TARGET_COL].isin([-1.0, 1.0])]
    d['y'] = (d[TARGET_COL] == 1).astype(int)
    d['date'] = pd.to_datetime(d['datetime'], utc=True).dt.date
    d['ret_next'] = d.groupby('session_key')['close_price'].pct_change().shift(-1)
    d = d.dropna(subset=['ret_next'])
    tr = d[d['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    va = d[d['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    te = d[d['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    return tr, va, te, d

def fit_eval_for_bands(name, model, feat_cols, tr, va, te):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(tr[feat_cols].fillna(0))
    Xva = scaler.transform(va[feat_cols].fillna(0))
    Xte = scaler.transform(te[feat_cols].fillna(0))
    ytr, yva, yte = tr['y'].values, va['y'].values, te['y'].values
    model.fit(Xtr, ytr)
    pva = model.predict_proba(Xva)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]
    return {'model': name, 'auc_val': roc_auc_score(yva, pva), 'auc_test': roc_auc_score(yte, pte), 'proba_test': pte}

def rows_for_threshold_bands(base_result, te_df, feature_set):
    rows = []
    for band, thr in THRESHOLD_BANDS.items():
        bt = backtest_pnl_bands(base_result['proba_test'], te_df['ret_next'].values, te_df['session_key'].values, threshold=thr)
        rows.append({'model': base_result['model'], 'feature_set': feature_set, 'threshold_band': band, 'threshold': thr,
                     'auc_val': base_result['auc_val'], 'auc_test': base_result['auc_test'], 'net_%': bt['net_%'],
                     'trades': bt['trades'], 'avg_%_per_trade': bt['avg_%_per_trade']})
    return rows

def fit_eval_backtest(name, model, X_train, y_train, X_val, y_val, X_test, y_test, test_df_bt, feat_cols):
    model.fit(X_train, y_train)
    auc_val = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    auc_test = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    proba = model.predict_proba(X_test)[:, 1]
    ret = test_df_bt['ret_next'].values
    sess = test_df_bt['session_key'].values
    bt = backtest_pnl(proba, ret, sess)
    return {'model': name, 'auc_val': auc_val, 'auc_test': auc_test, 'net_%': bt['net_%'], 'trades': bt['trades']}

## 4. Сложные ML на baseline-фичах

LogReg, RF, XGBoost, LightGBM, CatBoost на 22 фичах. StandardScaler fit на train.

In [4]:
scaler_base = StandardScaler()
X_train_b = scaler_base.fit_transform(train_df[BASE_FEATURES].fillna(0))
X_val_b = scaler_base.transform(val_df[BASE_FEATURES].fillna(0))
X_test_b = scaler_base.transform(test_df[BASE_FEATURES].fillna(0))
y_train, y_val, y_test = train_df['y'].values, val_df['y'].values, test_df['y'].values

models_base = [
    ('logreg', LogisticRegression(max_iter=1200, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)),
]
if HAS_XGB:
    models_base.append(('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')))
if HAS_LGB:
    models_base.append(('lgb', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)))
if HAS_CAT:
    models_base.append(('cat', cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)))

results_base = []
for name, m in models_base:
    X_tr = pd.DataFrame(X_train_b, columns=BASE_FEATURES)
    X_va = pd.DataFrame(X_val_b, columns=BASE_FEATURES)
    X_te = pd.DataFrame(X_test_b, columns=BASE_FEATURES)
    r = fit_eval_backtest(f'{name}_base', m, X_tr, y_train, X_va, y_val, X_te, y_test, test_df, BASE_FEATURES)
    r['features'] = 'baseline'
    results_base.append(r)

print('Baseline (22 фичи), порог 0.40–0.60:')
for r in sorted(results_base, key=lambda x: -x['net_%']):
    print(f"  {r['model']:12} AUC_val={r['auc_val']:.4f} AUC_test={r['auc_test']:.4f} net_PnL={r['net_%']:.2f}% trades={r['trades']}")

Baseline (22 фичи), порог 0.40–0.60:
  lgb_base     AUC_val=0.7554 AUC_test=0.6283 net_PnL=1013.16% trades=1762
  xgb_base     AUC_val=0.7531 AUC_test=0.6230 net_PnL=1008.83% trades=1937
  cat_base     AUC_val=0.7577 AUC_test=0.6266 net_PnL=911.06% trades=1416
  rf_base      AUC_val=0.7626 AUC_test=0.6180 net_PnL=822.81% trades=1181
  logreg_base  AUC_val=0.7220 AUC_test=0.6131 net_PnL=759.62% trades=837


## 5. Сложные ML на Sequence-фичах (rolling 30, 60)

Те же модели на 62 фичах (baseline + rolling mean/std по окнам 30, 60).

In [5]:
scaler_seq = StandardScaler()
X_train_s = scaler_seq.fit_transform(train_df_seq[FEATURES_SEQ].fillna(0))
X_val_s = scaler_seq.transform(val_df_seq[FEATURES_SEQ].fillna(0))
X_test_s = scaler_seq.transform(test_df_seq[FEATURES_SEQ].fillna(0))
y_train_s = train_df_seq['y'].values
y_val_s = val_df_seq['y'].values
y_test_s = test_df_seq['y'].values

models_seq = [
    ('logreg', LogisticRegression(max_iter=1200, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)),
]
if HAS_XGB:
    models_seq.append(('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')))
if HAS_LGB:
    models_seq.append(('lgb', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)))
if HAS_CAT:
    models_seq.append(('cat', cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)))

results_seq = []
for name, m in models_seq:
    X_tr = pd.DataFrame(X_train_s, columns=FEATURES_SEQ)
    X_va = pd.DataFrame(X_val_s, columns=FEATURES_SEQ)
    X_te = pd.DataFrame(X_test_s, columns=FEATURES_SEQ)
    r = fit_eval_backtest(f'{name}_seq', m, X_tr, y_train_s, X_va, y_val_s, X_te, y_test_s, test_df_seq, FEATURES_SEQ)
    r['features'] = 'sequence'
    results_seq.append(r)

print('Sequence (62 фичи, rolling 30,60), порог 0.40–0.60:')
for r in sorted(results_seq, key=lambda x: -x['net_%']):
    print(f"  {r['model']:12} AUC_val={r['auc_val']:.4f} AUC_test={r['auc_test']:.4f} net_PnL={r['net_%']:.2f}% trades={r['trades']}")

Sequence (62 фичи, rolling 30,60), порог 0.40–0.60:
  lgb_seq      AUC_val=0.7735 AUC_test=0.6288 net_PnL=1035.03% trades=1564
  xgb_seq      AUC_val=0.7769 AUC_test=0.6269 net_PnL=998.74% trades=1702
  cat_seq      AUC_val=0.7738 AUC_test=0.6340 net_PnL=867.23% trades=1235
  logreg_seq   AUC_val=0.7262 AUC_test=0.6199 net_PnL=843.70% trades=879
  rf_seq       AUC_val=0.7442 AUC_test=0.6175 net_PnL=714.66% trades=704


## 6. Модель с временными фичами (flatten sequence)

Подаём последние 60 баров baseline-фичей как один вектор (60 × 22 = 1320 dim). MLPClassifier — простой способ «увидеть» временную структуру без LSTM. Альтернатива — LSTM/Transformer при наличии PyTorch.

In [6]:
SEQ_LEN = 60

def build_temporal_arrays(df_subset, feat_cols, seq_len=SEQ_LEN):
    """Для каждой сессии: последние seq_len баров feat_cols → flatten. Возвращает X, y, ret, sess с выровненными длинами."""
    df_subset = df_subset.sort_values(['session_key', sort_col]).reset_index(drop=True)
    X_list, y_list, ret_list, sess_list = [], [], [], []
    for skey, g in df_subset.groupby('session_key', sort=False):
        g = g.dropna(subset=feat_cols + [TARGET_COL, 'ret_next']).copy()
        g = g[g[TARGET_COL].isin([-1.0, 1.0])]
        if len(g) <= seq_len:
            continue
        arr = g[feat_cols].fillna(0).values
        for i in range(seq_len, len(g)):
            X_list.append(arr[i - seq_len:i].flatten())
            y_list.append(1 if g.iloc[i][TARGET_COL] == 1.0 else 0)
            ret_list.append(g.iloc[i]['ret_next'])
            sess_list.append(skey)
    X = np.array(X_list) if X_list else np.empty((0, seq_len * len(feat_cols)))
    y = np.array(y_list) if y_list else np.array([], dtype=int)
    ret = np.array(ret_list) if ret_list else np.array([])
    sess = np.array(sess_list) if sess_list else np.array([])
    return X, y, ret, sess

df_temp = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
df_temp = df_temp[df_temp[TARGET_COL].isin([-1.0, 1.0])]
df_temp['ret_next'] = df_temp.groupby('session_key')['close_price'].pct_change().shift(-1)
df_temp['date'] = pd.to_datetime(df_temp['datetime'], utc=True).dt.date
df_temp = df_temp.dropna(subset=['ret_next'])

train_temp = df_temp[df_temp['date'].isin(train_dates)]
val_temp = df_temp[df_temp['date'].isin(val_dates)]
test_temp = df_temp[df_temp['date'].isin(test_dates)]

X_train_t, y_train_t, _, _ = build_temporal_arrays(train_temp, BASE_FEATURES, SEQ_LEN)
X_val_t, y_val_t, _, _ = build_temporal_arrays(val_temp, BASE_FEATURES, SEQ_LEN)
X_test_t, y_test_t, test_ret_t, test_sess_t = build_temporal_arrays(test_temp, BASE_FEATURES, SEQ_LEN)

if len(X_train_t) > 0 and len(y_train_t) > 0 and len(X_test_t) > 0:
    scaler_t = StandardScaler()
    X_train_t = scaler_t.fit_transform(X_train_t)
    X_val_t = scaler_t.transform(X_val_t)
    X_test_t = scaler_t.transform(X_test_t)
    mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=300, random_state=42, early_stopping=True, validation_fraction=0.1)
    mlp.fit(X_train_t, y_train_t)
    auc_val_t = roc_auc_score(y_val_t, mlp.predict_proba(X_val_t)[:, 1]) if len(y_val_t) > 0 else 0.5
    auc_test_t = roc_auc_score(y_test_t, mlp.predict_proba(X_test_t)[:, 1]) if len(y_test_t) > 0 else 0.5
    proba_t = mlp.predict_proba(X_test_t)[:, 1]
    bt_t = backtest_pnl(proba_t, test_ret_t, test_sess_t)
    results_temporal = [{'model': 'mlp_temporal', 'auc_val': auc_val_t, 'auc_test': auc_test_t, 'net_%': bt_t['net_%'], 'trades': bt_t['trades'], 'features': 'temporal_flatten'}]
    print('Temporal (MLP на flatten 60×22), порог 0.40–0.60:')
    print(f"  mlp_temporal  AUC_val={auc_val_t:.4f} AUC_test={auc_test_t:.4f} net_PnL={bt_t['net_%']:.2f}% trades={bt_t['trades']}")
else:
    results_temporal = []
    print('Temporal: недостаточно данных для последовательностей (пропуск)')

Temporal (MLP на flatten 60×22), порог 0.40–0.60:
  mlp_temporal  AUC_val=0.5288 AUC_test=0.5333 net_PnL=-30.61% trades=2483


## 7. Прод-модель на том же тесте

Загружаем `best_model.joblib` (CatBoost из шага 07), применяем на тестовом дне с порогом 0.40–0.60. Сравниваем с экспериментами выше.

In [7]:
prod_path = os.path.join(_root, 'models', 'best_model.joblib')
if os.path.exists(prod_path):
    prod = joblib.load(prod_path)
    prod_model = prod['model']
    prod_scaler = prod['scaler']
    prod_feat = prod['features']
    X_test_prod = prod_scaler.transform(test_df[prod_feat].fillna(0))
    proba_prod = prod_model.predict_proba(X_test_prod)[:, 1]
    ret = test_df['ret_next'].values
    sess = test_df['session_key'].values
    bt_prod = backtest_pnl(proba_prod, ret, sess)
    auc_test_prod = roc_auc_score(test_df['y'].values, proba_prod)
    results_prod = [{'model': 'prod_catboost', 'auc_val': np.nan, 'auc_test': auc_test_prod, 'net_%': bt_prod['net_%'], 'trades': bt_prod['trades'], 'features': 'baseline'}]
    print('Прод (best_model.joblib, CatBoost), порог 0.40–0.60:')
    print(f"  prod_catboost AUC_test={auc_test_prod:.4f} net_PnL={bt_prod['net_%']:.2f}% trades={bt_prod['trades']}")
else:
    results_prod = []
    print('best_model.joblib не найден — пропуск')

Прод (best_model.joblib, CatBoost), порог 0.40–0.60:
  prod_catboost AUC_test=0.6253 net_PnL=895.72% trades=1423


## 8. Итоговая таблица сравнения

Все модели на одном тестовом дне, порог 0.40–0.60. Ничего не сохраняем в прод.

In [8]:
all_results = results_base + results_seq + results_temporal + results_prod
summary = pd.DataFrame(all_results)[['model', 'features', 'auc_val', 'auc_test', 'net_%', 'trades']]
summary = summary.sort_values('net_%', ascending=False).reset_index(drop=True)
print('Сравнение (порог 0.40–0.60, temporal split, один тестовый день):')
print(summary.to_string(index=False))
print()
print('Лучший по net PnL:', summary.iloc[0]['model'], f"({summary.iloc[0]['net_%']:.2f}%)")
print('(Ничего не сохранено в прод — только эксперимент)')

Сравнение (порог 0.40–0.60, temporal split, один тестовый день):
        model         features  auc_val  auc_test       net_%  trades
      lgb_seq         sequence 0.773493  0.628828 1035.034743    1564
     lgb_base         baseline 0.755425  0.628289 1013.157620    1762
     xgb_base         baseline 0.753120  0.623011 1008.829117    1937
      xgb_seq         sequence 0.776894  0.626949  998.741472    1702
     cat_base         baseline 0.757723  0.626608  911.058399    1416
prod_catboost         baseline      NaN  0.625301  895.723099    1423
      cat_seq         sequence 0.773804  0.633961  867.232568    1235
   logreg_seq         sequence 0.726230  0.619889  843.698805     879
      rf_base         baseline 0.762590  0.617960  822.814610    1181
  logreg_base         baseline 0.722027  0.613062  759.620119     837
       rf_seq         sequence 0.744184  0.617549  714.662020     704
 mlp_temporal temporal_flatten 0.528786  0.533327  -30.610463    2483

Лучший по net PnL: lgb_s

## Итог

- Сравнили сложные ML (LogReg, RF, XGB, LGB, CatBoost) на baseline и Sequence (rolling 30, 60).
- Добавили MLP на flatten-последовательности (60×22) как простой «temporal» вариант.
- Прод-модель (CatBoost) оценена на том же тесте с порогом 0.40–0.60.

### Выводы

1. **Sequence (rolling 30, 60) > Baseline** — rolling mean/std дают прирост: lgb_seq лучше lgb_base, xgb_seq лучше xgb_base.
2. **Бустинги (LGB, XGB, CatBoost) > RF, LogReg** — по net PnL лидируют градиентные бустинги.
3. **MLP на flatten — провал** — AUC_val≈0.53, AUC_test≈0.53, net_PnL≈−31%. Flatten без явной временной структуры не работает.
4. **Val vs Test** — AUC_val выше AUC_test у всех; есть переобучение, но test-метрики в разумных пределах (~0.61–0.63).

### Кандидат для прода: lgb_seq

**lgb_seq** (LightGBM на sequence-фичах) — лучший по net PnL в этом эксперименте:

- **AUC_val=0.77, AUC_test=0.63** — стабильная генерализация.
- **net_PnL=1035%** — выше lgb_base (1013%), xgb_seq (999%) и prod CatBoost (896%).
- **trades=1564** — умеренная активность, меньше, чем у xgb_base (1937).

**Особенности:** 62 фичи (baseline 22 + rolling mean/std по окнам 30, 60), порог 0.40–0.60, комиссия 0.1% round-trip. Стоит рассмотреть для проды после дополнительной валидации на новых тестовых днях и A/B-теста.

## 9. Дополнительный эксперимент: rolling 5/15/30/60 vs 30/60

Использует данные и функции из блоков 1–3 (выполните **Run All**).

- Сравниваем `old_30_60` и `new_5_15_30_60` на том же temporal split и комиссии 0.1%.
- Пороги: 20-80, 25-75, 30-70, 35-65, 40-60.
- Метрики: auc_val, auc_test, net_%, trades, avg_%_per_trade.

In [2]:
# Секция 9: использует df, BASE_FEATURES, train_dates, THRESHOLD_BANDS и функции из блоков 1–3.
# Выполняйте Run All перед секцией 9.

Loaded: 344,415 rows, 22 features, train=2026-02-01..2026-02-08, val=2026-02-09, test=2026-02-10


In [3]:
# Формируем old/new наборы rolling-фичей
key_feats_15 = BASE_FEATURES[:10]
df_feat_15 = df.copy()
grp_15 = df_feat_15.groupby('session_key', group_keys=False)
for w in [5, 15, 30, 60]:
    for c in key_feats_15:
        if c in df_feat_15.columns:
            df_feat_15[f'{c}_roll{w}_mean'] = grp_15[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df_feat_15[f'{c}_roll{w}_std'] = grp_15[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

old_cols_15 = [c for c in df_feat_15.columns if ('_roll30_' in c or '_roll60_' in c)]
new_cols_15 = [c for c in df_feat_15.columns if ('_roll5_' in c or '_roll15_' in c or '_roll30_' in c or '_roll60_' in c)]
FEATURES_OLD_15 = [c for c in (BASE_FEATURES + old_cols_15) if c in df_feat_15.columns]
FEATURES_NEW_15 = [c for c in (BASE_FEATURES + new_cols_15) if c in df_feat_15.columns]

tr_old_15, va_old_15, te_old_15, all_old_15 = prep_by_features(df_feat_15, FEATURES_OLD_15)
tr_new_15, va_new_15, te_new_15, all_new_15 = prep_by_features(df_feat_15, FEATURES_NEW_15)

print('OLD features:', len(FEATURES_OLD_15), '| NEW features:', len(FEATURES_NEW_15))

# Обучение old/new и сравнение по всем band'ам
res_old_15, res_new_15 = [], []

models_15 = [
    ('logreg', LogisticRegression(max_iter=1200, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)),
]
if HAS_XGB:
    models_15.append(('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')))
if HAS_LGB:
    models_15.append(('lgb', lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)))
if HAS_CAT:
    models_15.append(('cat', cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)))

for n, m in models_15:
    base = fit_eval_for_bands(f'{n}_seq_old', m, FEATURES_OLD_15, tr_old_15, va_old_15, te_old_15)
    res_old_15.extend(rows_for_threshold_bands(base, te_old_15, 'old_30_60'))

for n, m in models_15:
    base = fit_eval_for_bands(f'{n}_seq_new', m, FEATURES_NEW_15, tr_new_15, va_new_15, te_new_15)
    res_new_15.extend(rows_for_threshold_bands(base, te_new_15, 'new_5_15_30_60'))

summary_15 = pd.DataFrame(res_old_15 + res_new_15)[['model', 'feature_set', 'threshold_band', 'threshold', 'auc_val', 'auc_test', 'net_%', 'trades', 'avg_%_per_trade']]
summary_15 = summary_15.sort_values(['model', 'threshold_band', 'net_%'], ascending=[True, True, False]).reset_index(drop=True)

print('\n=== Compare old vs new (20-80, 25-75, 30-70, 35-65, 40-60) ===')
print(summary_15.to_string(index=False))

OLD features: 62 | NEW features: 102

=== Compare old vs new (20-80, 25-75, 30-70, 35-65, 40-60) ===
         model    feature_set threshold_band  threshold  auc_val  auc_test       net_%  trades  avg_%_per_trade
   cat_seq_new new_5_15_30_60          20-80       0.80 0.788630  0.641064  308.725534     128         2.411918
   cat_seq_new new_5_15_30_60          25-75       0.75 0.788630  0.641064  419.850844     239         1.756698
   cat_seq_new new_5_15_30_60          30-70       0.70 0.788630  0.641064  516.322827     393         1.313799
   cat_seq_new new_5_15_30_60          35-65       0.65 0.788630  0.641064  685.346005     732         0.936265
   cat_seq_new new_5_15_30_60          40-60       0.60 0.788630  0.641064  980.411715    1297         0.755907
   cat_seq_old      old_30_60          20-80       0.80 0.773804  0.633961  250.283016     114         2.195465
   cat_seq_old      old_30_60          25-75       0.75 0.773804  0.633961  353.356794     200         1.766784
   

In [ ]:
# Стабильность по дням для LightGBM: baseline vs old vs new и обе логики сделок
if HAS_LGB:
    eval_days_15 = [d for d in dates if d not in train_dates]

    def fit_lgb_daywise_15(feat_cols, label):
        d = df_feat_15.dropna(subset=feat_cols + [TARGET_COL]).copy()
        d = d[d[TARGET_COL].isin([-1.0, 1.0])]
        d['y'] = (d[TARGET_COL] == 1).astype(int)
        d['date'] = pd.to_datetime(d['datetime'], utc=True).dt.date
        d['ret_next'] = d.groupby('session_key')['close_price'].pct_change().shift(-1)
        d = d.dropna(subset=['ret_next'])

        tr = d[d['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(tr[feat_cols].fillna(0))
        ytr = tr['y'].values
        model = lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
        model.fit(Xtr, ytr)

        rows = []
        for day in eval_days_15:
            dd = d[d['date'] == day].sort_values(['session_key', sort_col]).reset_index(drop=True)
            if len(dd) == 0:
                continue
            Xd = scaler.transform(dd[feat_cols].fillna(0))
            p = model.predict_proba(Xd)[:, 1]
            auc = roc_auc_score(dd['y'].values, p)
            for band, thr in THRESHOLD_BANDS.items():
                bt_signal = backtest_pnl(p, dd['ret_next'].values, dd['session_key'].values, threshold=thr, hold_keep_prev=True)
                bt_05 = backtest_entry_exit_05(p, dd['ret_next'].values, dd['session_key'].values, threshold=thr)
                rows.append({'day': day, 'variant': label, 'logic': 'signal_flip', 'threshold_band': band, 'threshold': thr, 'auc': auc, 'net_%': bt_signal['net_%'], 'trades': bt_signal['trades'], 'avg_%_per_trade': bt_signal['avg_%_per_trade']})
                rows.append({'day': day, 'variant': label, 'logic': 'exit_on_05', 'threshold_band': band, 'threshold': thr, 'auc': auc, 'net_%': bt_05['net_%'], 'trades': bt_05['trades'], 'avg_%_per_trade': bt_05['avg_%_per_trade']})
        return rows

    stab_15 = pd.DataFrame(
        fit_lgb_daywise_15(FEATURES_BASE_15, 'baseline_22') +
        fit_lgb_daywise_15(FEATURES_OLD_15, 'old_30_60') +
        fit_lgb_daywise_15(FEATURES_NEW_15, 'new_5_15_30_60')
    )
    stab_15 = stab_15.sort_values(['day', 'variant', 'logic', 'threshold_band']).reset_index(drop=True)
    print('=== Day-by-day stability (lgb, feature windows + two logics) ===')
    print(stab_15.to_string(index=False))

    agg_15 = stab_15.groupby(['variant', 'logic', 'threshold_band'], as_index=False).agg(
        days=('day', 'count'),
        auc_mean=('auc', 'mean'),
        net_mean=('net_%', 'mean'),
        net_median=('net_%', 'median'),
        trades_mean=('trades', 'mean'),
        avg_trade_mean=('avg_%_per_trade', 'mean')
    )
    print('\n=== Stability aggregate ===')
    print(agg_15.to_string(index=False))
else:
    print('LightGBM не установлен, блок стабильности пропущен.')